# 03 — Universe master feature table

Build one **pooled labeled table** from all available symbols for a selected interval, plus a separate **latest-bar inference table**. This notebook does not train a predictive model.

**Leakage rules:** features are computed independently for each ticker; targets never cross ticker boundaries; quantile boundaries, K-means, and preprocessing are fitted on the earlier chronological training period only. The final bar per ticker has no known next-bar outcome and is reserved for future inference.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.db import engine
from stock_ml_dynamic.master_table import (fetch_universe_prices, build_universe_features, assemble_master_table, model_ready, save_master_outputs, FEATURE_COLUMNS, NUMERIC_FEATURES, CATEGORICAL_FEATURES)


## 1. Settings

`SYMBOLS=None` means all stocks present for this interval (expected: 220). Start with `1d`; run separately for intraday/weekly/monthly intervals. `BARS_PER_SYMBOL` limits the most recent bars fetched per stock. Use `SELECTED_K=None` for the elbow heuristic, or an integer from 2 to 8.

In [ ]:
INTERVAL = "1d"
BARS_PER_SYMBOL = 2000
SYMBOLS = None  # or ["AAPL", "MSFT"] for a quick trial
TRAIN_FRACTION = 0.80
Z_WINDOW = 30
SELECTED_K = None
SAVE_RESULTS = True


## 2. Read the universe from PostgreSQL

One read-only SQL query obtains up to `BARS_PER_SYMBOL` bars **per ticker**, not globally.

In [ ]:
with engine().connect() as conn:
    prices = fetch_universe_prices(conn, interval=INTERVAL, bars_per_symbol=BARS_PER_SYMBOL, symbols=SYMBOLS)
print(f"Raw bars: {len(prices):,}; symbols: {prices.symbol.nunique()}; interval: {INTERVAL}")
display(prices.groupby("symbol").size().describe().to_frame("bars_per_symbol"))
display(prices.head())


## 3. Build causal features and next-bar outcomes

`direction` is still unknown here: it will be assigned using **training-only quantile boundaries**. `next_adj_close` and `next_return` are computed separately within each stock.

In [ ]:
features = build_universe_features(prices, z_window=Z_WINDOW)
print("Feature rows:", len(features))
display(features[["symbol", "bar_timestamp"] + NUMERIC_FEATURES + ["next_adj_close", "next_return"]].tail(8))


## 4. Fit training-only target bins, K-means candle types, and preprocessing

The same fitted transforms will later be used for every stock and for live predictions. The global chronological cutoff avoids fitting on future timestamps of other tickers.

In [ ]:
result = assemble_master_table(features, train_fraction=TRAIN_FRACTION, selected_k=SELECTED_K)
print("Train cutoff:", result.cutoff)
print("Selected K:", result.clusters.chosen_k_)
print("Quantile edges:", result.quantiles.edges_)
display(result.clusters.inertia_)
result.clusters.inertia_.plot(x="k", y="inertia", marker="o", title="Pooled candle K-means elbow", legend=False)
plt.show()


## 5. Inspect the labeled master table and latest-bar inference rows

There is one latest bar per ticker with **unknown direction and next adjusted close**. Those rows are intentionally absent from `master` but retained in `inference`.

In [ ]:
master = result.master
inference = result.inference
cols = ["symbol", "interval_code", "bar_timestamp"] + FEATURE_COLUMNS + ["direction", "adj_close", "next_adj_close", "next_return"]
print(f"Master: {len(master):,} labeled rows across {master.symbol.nunique()} stocks")
print(f"Inference: {len(inference):,} latest rows across {inference.symbol.nunique()} stocks")
display(master[cols].head(10))
display(inference[cols].tail(10))
assert master.direction.notna().all()
assert inference.direction.isna().all()
assert inference.next_adj_close.isna().all()


## 6. Class distribution and missingness

Class IDs: **2 = down, 0 = no_change, 1 = up**. With pooled quantiles, global training classes are approximately balanced, but individual stocks can differ.

In [ ]:
display(master.direction.value_counts(dropna=False).rename_axis("direction").to_frame("rows"))
display(master.groupby("symbol").size().describe().to_frame("labeled_rows_per_symbol"))
display(master[FEATURE_COLUMNS].isna().sum().to_frame("missing_feature_rows"))


## 7. Model-ready matrices

`X_master` is numerically standardized and categoricals are one-hot encoded using **training-only fitted preprocessing**. Use `train_rows` to select training data and `~train_rows` for chronological validation. **Do not refit** the scaler/encoder on test or inference data. For regression, `next_adj_close` is the next-bar target, while `adj_close` is the current observed price.

In [ ]:
X_master, aligned_master = model_ready(result, "master")
X_latest, aligned_latest = model_ready(result, "inference")
y_class = aligned_master.direction.astype(int)
y_reg = aligned_master.next_adj_close.astype(float)
train_rows = aligned_master.next_bar_timestamp <= result.cutoff
print("X master:", X_master.shape, "X latest:", X_latest.shape)
print("Training:", int(train_rows.sum()), "Validation:", int((~train_rows).sum()))
print("Regression labels:", len(y_reg), "Classification labels:", len(y_class))
display(X_master.head())


## 8. Persist artifacts for future training and Streamlit inference

The saved `transformers.joblib` contains the fitted quantile boundaries, candle cluster model/scaler, feature normalizer, and one-hot encoder. Future model training must use these exact transforms and the recorded feature order. Parquet files are local artifacts and should not be committed if large.

In [ ]:
if SAVE_RESULTS:
    output_dir = save_master_outputs(result, interval=INTERVAL)
    print("Saved to:", output_dir)
    print("Files:", [p.name for p in output_dir.iterdir()])
else:
    print("Saving disabled")


## 9. Optional multi-step regression target preview

For a future regression trajectory, create `adj_close_t+1` through `adj_close_t+15` **per ticker**. The final 15 rows of each stock lack some future outcomes and must be excluded from training a 15-output regressor, but preserved for inference.


In [ ]:
from stock_ml_dynamic.master_table import add_forward_price_targets
forward = add_forward_price_targets(features, horizons=range(1, 16))
future_columns = [f"adj_close_t+{h}" for h in range(1, 16)]
display(forward.loc[forward.symbol == forward.symbol.iloc[0], ["symbol", "bar_timestamp", "adj_close"] + future_columns].tail(18))
multi_output_labeled = forward.dropna(subset=future_columns)
print("Fully labeled 15-step regression rows:", len(multi_output_labeled))


## Next notebook: classifiers and regressors

Build a **chronological** classifier using `X_master`, `y_class` and `train_rows`. Use multiclass ROC-AUC (one-vs-rest) only when all needed classes are present. For regression, use `y_reg` and consider predicting future returns instead of raw prices to make cross-stock learning more stable. To predict **multiple future bars**, construct `next_adj_close_t+1` through `t+N` **within each ticker**, never using a global shift. Keep the latest-bar `X_latest` for deployment. K-means labels and encodings must stay consistent with the saved transformers.